<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session12-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 12: in-class discussion problem (2 of 3)

This is the block 2 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then run the code below and look at what it does. There is one mistake in it, but finding that is
only the way in: the point is to work out why the result comes out as it does. Discuss that with
your neighbour, then answer the quiz again.

## What does an average pLDDT hide?

AlphaFold stores its per-residue confidence, pLDDT (0–100), in the B-factor column of the model
file. MDM2 is the protein that binds p53's transactivation domain and marks p53 for degradation.
The AlphaFold DB model of full-length human MDM2 (UniProt Q00987, 491 residues) has a mean pLDDT of
63: "low" on AlphaFold's own scale, which calls 70–90 confident and above 90 very high.

What is the mean pLDDT of its p53-binding domain, residues 25–109?

**Fix and run.** The cell below downloads the model and prints the mean pLDDT of the whole chain
and of the p53-binding domain. The line marked `FIX` averages the wrong array. Run it once as it
is, then fix it and run it again.

In [1]:
# Install the libraries Colab does not come with (does nothing if they are already installed)
import importlib.util, subprocess, sys
for module, package in [("Bio", "biopython")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [2]:
import io
import numpy as np, requests
from Bio.PDB import PDBParser

meta = requests.get("https://alphafold.ebi.ac.uk/api/prediction/Q00987", timeout=60).json()[0]
text = requests.get(meta["pdbUrl"], timeout=60).text
chain = next(iter(PDBParser(QUIET=True).get_structure("MDM2", io.StringIO(text))[0]))
plddt = np.array([r["CA"].get_bfactor() for r in chain])     # one value per residue, residue 1 first

print(f"whole chain, {len(plddt)} residues: mean pLDDT {plddt.mean():.1f}")
domain = plddt[24:109]                                         # residues 25-109 (Python counts from 0)
print(f"p53-binding domain, {len(domain)} residues: mean pLDDT {plddt.mean():.1f}")   # FIX: which array?

whole chain, 491 residues: mean pLDDT 62.6
p53-binding domain, 85 residues: mean pLDDT 62.6


**If you finish early.**

1. Print the mean pLDDT of residues 110–429. What do you expect for this part of MDM2?
2. Plot `plddt` along the sequence with `import matplotlib.pyplot as plt; plt.plot(plddt)`. Which
   other stretches are confident? (UniProt: a zinc finger around 290–330, a RING domain at the
   C-terminus.)
3. Low pLDDT often marks disorder, but not always. Name another reason a region can get a low pLDDT.

<details>
<summary><b>Discussion point</b> &mdash; open this once you have run the code and talked it through</summary>

As it was, the second line printed the mean of `plddt`, the whole chain,
again: 62.6. The fix averages the domain:

```python
print(f"p53-binding domain, {len(domain)} residues: mean pLDDT {domain.mean():.1f}")
```

The p53-binding domain then has a mean pLDDT of 94.7: very high. The middle of MDM2, residues
110–429, averages about 50, apart from a short zinc finger; the RING domain at the C-terminus is
confident again. A mean of 63 is not "a mediocre model everywhere": it is confident domains joined
by long stretches AlphaFold does not place, in MDM2 mostly disordered. The same holds for p53 itself
(the lecture). Always look at pLDDT along the sequence, and at the PAE between domains, never only at
the mean.

</details>